In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
import warnings

warnings.filterwarnings('ignore')

# --- 1. Load Data (Keep Duplicates) ---
# As confirmed, duplicates are part of the test set structure. We keep them.
train = pd.read_csv('/kaggle/input/playground-series-s5e12/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')

train = train.drop(columns=['id'])
test_ids = test['id']
test = test.drop(columns=['id'])

# --- 2. Feature Engineering & Clustering ---
def process_data(df_train, df_test):
    # Combine for consistent processing
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    full = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # A. Ordinal Mapping (Hardcoded for Medical Logic)
    # Don't let the model guess; tell it that "Master's" > "High School"
    edu_map = {
        'Less than High School': 0, 'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4
    }
    inc_map = {
        'Under $15k': 0, '$15k-$25k': 1, '$25k-$35k': 2, '$35k-$50k': 3,
        '$50k-$75k': 4, '$75k-$100k': 5, 'Over $100k': 6
    }
    
    if 'education_level' in full.columns:
        full['education_level'] = full['education_level'].map(edu_map).fillna(1) # Fill mode
    if 'income_level' in full.columns:
        full['income_level'] = full['income_level'].map(inc_map).fillna(3) # Fill mode
        
    # B. Medical Interactions
    full['BMI_Waist'] = full['bmi'] * full['waist_to_hip_ratio']
    full['Lipid_Ratio'] = full['triglycerides'] / (full['hdl_cholesterol'] + 1)
    full['BP_Stress'] = (full['systolic_bp'] - full['diastolic_bp']) * full['heart_rate']
    
    # C. K-MEANS CLUSTERING (The New Feature)
    # We cluster patients based on their "Vitals" only.
    # This creates a "Patient Type" feature (e.g., Cluster 3 might be "High BP, Low BMI").
    cluster_cols = ['bmi', 'systolic_bp', 'glucose', 'cholesterol_total', 'age']
    # Filter only columns that exist
    use_cols = [c for c in cluster_cols if c in full.columns]
    
    # Scale before clustering
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(full[use_cols].fillna(0))
    
    # Create 7 Clusters (Experimentally good number for medical profiles)
    kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
    full['Patient_Cluster'] = kmeans.fit_predict(scaled_data)
    
    # Split back
    train_processed = full[full['is_train'] == 1].drop(columns=['is_train'])
    test_processed = full[full['is_train'] == 0].drop(columns=['is_train'])
    
    return train_processed, test_processed

X, X_test = process_data(train, test)

y = X['diagnosed_diabetes']
X = X.drop(columns=['diagnosed_diabetes'])
X_test = X_test.drop(columns=['diagnosed_diabetes'])

# --- 3. Encoding ---
# We use Label Encoding for the remaining categoricals (Ethnicity, Gender, etc.)
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
# Add Cluster to cat_cols so CatBoost treats it properly
cat_cols.append('Patient_Cluster') 

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# --- 4. Training (Weighted Ensemble) ---
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Arrays to store predictions
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

print("Starting Cluster-Enhanced Training...")

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[trn_idx], y.iloc[trn_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # --- Model A: XGBoost ---
    model_xgb = xgb.XGBClassifier(
        n_estimators=2000,
        learning_rate=0.015,
        max_depth=5,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_lambda=3.0,
        tree_method='hist',
        random_state=42 + fold,
        n_jobs=-1
    )
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False, early_stopping_rounds=100)
    p_xgb = model_xgb.predict_proba(X_val)[:, 1]
    t_xgb = model_xgb.predict_proba(X_test)[:, 1]

    # --- Model B: LightGBM ---
    model_lgb = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.015,
        num_leaves=31,
        max_depth=6,
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_samples=50, # Force it to look at larger groups
        random_state=42 + fold,
        n_jobs=-1,
        verbose=-1
    )
    model_lgb.fit(X_train, y_train)
    p_lgb = model_lgb.predict_proba(X_val)[:, 1]
    t_lgb = model_lgb.predict_proba(X_test)[:, 1]
    
    # --- Model C: CatBoost ---
    # CatBoost handles the new 'Patient_Cluster' feature exceptionally well
    model_cb = cb.CatBoostClassifier(
        iterations=2000,
        learning_rate=0.015,
        depth=6,
        l2_leaf_reg=5,
        cat_features=cat_cols, # Explicitly tell it about clusters
        random_seed=42 + fold,
        verbose=False,
        allow_writing_files=False
    )
    model_cb.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    p_cb = model_cb.predict_proba(X_val)[:, 1]
    t_cb = model_cb.predict_proba(X_test)[:, 1]
    
    # --- Weighted Blend ---
    # CatBoost is usually strongest on Categorical data (Clusters), so we give it slightly more weight.
    # Weights: XGB (0.3), LGB (0.3), CB (0.4)
    oof_preds[val_idx] = (0.3 * p_xgb) + (0.3 * p_lgb) + (0.4 * p_cb)
    test_preds += ((0.3 * t_xgb) + (0.3 * t_lgb) + (0.4 * t_cb)) / 10
    
    print(f"Fold {fold+1} AUC: {roc_auc_score(y_val, oof_preds[val_idx]):.5f}")

# --- 5. Submission ---
overall_auc = roc_auc_score(y, oof_preds)
print(f"\n==============================")
print(f"Cluster Ensemble CV AUC: {overall_auc:.5f}")
print(f"==============================")

submission = pd.DataFrame({'id': test_ids, 'diagnosed_diabetes': test_preds})
submission.to_csv('submission.csv', index=False)
print("Submission saved!")

Starting Cluster-Enhanced Training...
Fold 1 AUC: 0.72499
Fold 2 AUC: 0.72626
Fold 3 AUC: 0.72370
Fold 4 AUC: 0.72343
Fold 5 AUC: 0.72465
Fold 6 AUC: 0.72427
Fold 7 AUC: 0.72324
Fold 8 AUC: 0.72781
Fold 9 AUC: 0.72711
Fold 10 AUC: 0.72360

Cluster Ensemble CV AUC: 0.72490
Submission saved!
